In [1]:
import numpy as np
from price_segmenter_v10 import *
from juejing import *
from tdx_quant import *

In [ ]:
# ============================================================
# Main
# ============================================================
def main():
    stocks = [
        # ('688017', '20251223', 250),
        # ('600550', '20200305', 200),
        # ('300437', '20210910', 150),
        # ('300437', '20251120', 200),
        # ('600199', '20200721', 200),
        # ('300204', '20250520', 200),
        # ('300204', '20250630', 200),
        # ('300171', '20260720', 200),
        # ('688017', '20260620', 200),
        # ('601179', '20251010', 200),
        # ('300251', '20250205', 150),
        # ('600403', '20251115', 200),
        # ('003036', '20261115', 200),
        # ('688017', '20260409', 350),
        # ('301313', '20260609', 150),
        # ('600578', '20260628', 150),
        # ('002552', '20260628', 250),
        # ('603283', '20260628', 250),
        # ('688110', '20250828', 250),
        # ('300408', '20251228', 250),
        # ('600246', '20260728', 250),
        # ('603629', '20260228', 250),
        # ('300546', '20250625', 250),
        # ('601958', '20260625', 200),
        # ('002842', '20260215', 200),
        # ('600378', '20260625', 200),
        # ('603629', '20260225', 200),
        # ('603373', '20250825', 200),
        # ('301176', '20260625', 200),
        # ('600176', '20260625', 300),
        # ('603256', '20260625', 300),
        # ('300437', '20251225', 150),
        # ('300437', '20210910', 150),
        # ('688387', '20250915', 150),
        ('600353', '20260915', 150),
        # ('300085', '20241015', 150),
        # ('003009', '20260128', 150),
        # ('000547', '20251128', 150),
        # ('601179', '20260128', 150),
        # ('300033', '20250928', 150),
        # ('300975', '20260828', 150),
        # ('600909', '20260828', 150),
        # ('000066', '20241228', 250),
        # ('300171', '20261228', 250),
        # ('300204', '20250728', 150),
    ]

    for code, end_date, tail in stocks:
        print(f"\n{'='*60}")
        print(f"  {code}  end={end_date}  tail={tail}")
        print(f"{'='*60}")

        # df = get_stock_klines_from_juejing(code, end_date, fqt=1)
        # df = df[df['close'] > 0].reset_index(drop=True)
        # df['date'] = pd.to_datetime(df['date'])
        df = get_daily_kline_from_tdx(code, end_date)

        run_segmentation(df, tail_days=tail, name=code,fast_mode=False)

if __name__ == "__main__":
    main()

In [ ]:
"""稳定获取「关联个股 / 同行业个股」F10 数据。

背景：TDX 不同行情主机返回的 F10 结构不一样——
  - 旧版主机（如 180.153.18.170）：返回 ~16 个分类，含「关联个股」（内含
    同行业个股、同地域个股、集团关联、基金持仓等多张表）。
  - 新版主机（如 124.70.x）：只返回 1 条「最新提示」，没有关联个股栏。
`from_best_host()` 每次按延迟随机挑主机，因此会出现「刚刚有、现在没了」。

解决：重试若干次，直到连上返回多分类（含关联个股）的主机；也可用
KNOWN_OLD_HOSTS 直接指定已知旧版主机优先尝试。
"""

from easy_tdx import Market, TdxClient

CODE = "603986"
MARKET = Market.SH
RELATED_KW = "关联个股"      # 该分类内含「同行业个股」等表
MAX_RETRY = 20               # 随机重连上限
MIN_CATS = 10                # 判定为旧版多分类主机的阈值
# 已知含关联个股的旧版主机（优先直连，命中即省去重试）
KNOWN_OLD_HOSTS = ["180.153.18.170"]


def _read_related(c):
    """在已连接的 client 上读取关联个股分类完整内容；无该分类返回 None。"""
    cats = c.get_company_info_category(MARKET, CODE)
    if len(cats) < MIN_CATS:
        return None, cats
    row = cats[cats["name"].str.contains(RELATED_KW, na=False)]
    if row.empty:
        return None, cats
    r = row.iloc[0]
    content = c.get_company_info_content(
        MARKET, CODE, str(r["filename"]), int(r["start"]), int(r["length"]))
    return content, cats


def get_related_stocks(code=CODE, market=MARKET):
    """稳定返回「关联个股」分类全文（含同行业个股表）。连不到旧版主机则抛异常。"""
    # 1) 优先直连已知旧版主机
    for host in KNOWN_OLD_HOSTS:
        try:
            with TdxClient(host=host) as c:
                content, cats = _read_related(c)
                if content:
                    return content, host, cats
        except Exception:
            pass
    # 2) 回退：重试 from_best_host 直到连上多分类主机
    for _ in range(MAX_RETRY):
        try:
            with TdxClient.from_best_host() as c:
                content, cats = _read_related(c)
                if content:
                    host = getattr(c, "_host", "?")
                    return content, host, cats
        except Exception:
            continue
    raise RuntimeError(
        f"重试 {MAX_RETRY} 次仍未连上含「{RELATED_KW}」的旧版主机——"
        f"当前多为只返回「最新提示」的新版主机，稍后重试或补充 KNOWN_OLD_HOSTS。")


if __name__ == "__main__":
    content, host, cats = get_related_stocks()
    print(f"命中主机 host={host}  目录条数={len(cats)}")
    print("分类列表:", cats["name"].tolist())
    print("=" * 64)
    print(content)


命中主机 host=180.153.18.170  目录条数=16
分类列表: ['最新提示', '公司概况', '财务分析', '股东研究', '股本结构', '资本运作', '业内点评', '行业分析', '公司大事', '研究报告', '经营分析', '主力追踪', '分红扩股', '高层治理', '龙虎榜单', '关联个股']
☆关联个股☆ ◇300171 东富龙 更新日期：2026-07-16◇ 港澳资讯 灵通V9.0
★本栏包括【1.同大股东个股】【2.同行业个股】【3.股本相近个股】【4.同地域个股】★

【1.同大股东个股】截止日期：2021-03-31
【华夏人寿保险股份有限公司－万能产品】（共3家）
排名    证券代码    股票名称  每股收益(元)  每股净资产(元)        营业收入(元)          营业利润(元)            净利润(元)
─────────────────────────────────────────────────────────────
1        300171      东富龙           0.18            5.92            7.1661亿              1.4243亿              1.1046亿
2        300184     力源信息          0.06            2.54           27.4885亿           8601.6846万           7027.0648万
3        002638     *ST勤上           0.00            2.38            3.3981亿          -1620.5938万            153.0181万
─────────────────────────────────────────────────────────────

【中国建设银行股份有限公司－中欧明睿新常态混合型证券投资基金】（共2家）
排名    证券代码    股票名称  每股收益(元)  每股净资产(元)        营业收入(元)          营业利润(元)            净利润(